In [ ]:
import pandas as pd

df = pd.read_csv("boat_data.csv", dtype=str, keep_default_na=False)

def fix(text):
    try:
        return text.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text

for col in df.columns:
    df[col] = df[col].map(fix)

print(df["Location"].head(6))

df.to_csv("boat_data_clean.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("boat_data_clean.csv")

0        Switzerland » Lake Geneva » Vésenaz
1                     Germany » Bönningstedt
2    Switzerland » Lake of Zurich » Stäfa ZH
3                        Denmark » Svendborg
4                 Germany » Bayern » München
5      Switzerland » Lake Constance » Uttwil
Name: Location, dtype: object


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
df["Currency"] = df["Price"].str.split(" ").str[0]
df["Amount"] = df["Price"].str.split(" ").str[1].astype(float)

print(df["Currency"].value_counts())
print(df["Amount"].describe())

Currency
EUR    8430
CHF     980
£       298
DKK     180
Name: count, dtype: int64
count    9.888000e+03
mean     3.201373e+05
std      1.007482e+06
min      3.300000e+03
25%      4.400000e+04
50%      9.500000e+04
75%      2.550000e+05
max      3.100000e+07
Name: Amount, dtype: float64


In [ ]:
rates = {"EUR": 1, "CHF": 0.925, "£": 1.163, "DKK": 0.1345}

df["Price_EUR"] = (df["Amount"] * df["Currency"].map(rates)).round(0)

print(df[["Price", "Price_EUR"]].head(8))
print(df["Price_EUR"].describe())

       Price  Price_EUR
0   CHF 3337     3087.0
1   EUR 3490     3490.0
2   CHF 3770     3487.0
3  DKK 25900     3484.0
4   EUR 3399     3399.0
5   CHF 3650     3376.0
6   CHF 3600     3330.0
7  DKK 24800     3336.0
count    9.888000e+03
mean     3.022844e+05
std      9.708675e+05
min      3.087000e+03
25%      4.250000e+04
50%      9.055750e+04
75%      2.462500e+05
max      3.100000e+07
Name: Price_EUR, dtype: float64


In [ ]:
df["Year Built"] = df["Year Built"].astype(int).replace(0, pd.NA).astype("Int64")

for col in ["Length", "Width", "Number of views last 7 days"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df["Year Built"].describe())
print("Yılı bilinmeyen ilan sayısı:", df["Year Built"].isna().sum())

count         9337.0
mean     2004.914962
std        16.356241
min           1885.0
25%           1999.0
50%           2008.0
75%           2018.0
max           2021.0
Name: Year Built, dtype: Float64
Yılı bilinmeyen ilan sayısı: 551


In [7]:
old = df[df["Year Built"] < 1950]
print(old["Material"].value_counts())
print(old["Boat Type"].value_counts().head(5))

Material
Steel    113
Wood      14
           2
GRP        2
Name: count, dtype: int64
Boat Type
Trawler           43
House Boat        22
Motor Yacht       20
Working Boat      11
Passenger boat     8
Name: count, dtype: int64


In [ ]:
df["Country"] = df["Location"].str.split("»").str[0].str.strip()

df["Country"] = df["Country"].replace({"Croatia (Hrvatska)": "Croatia"})

counts = df["Country"].value_counts()

main = counts[counts >= 35].index.drop("")

df["Country_Group"] = df["Country"].where(df["Country"].isin(main), "Other")

print(df["Country_Group"].value_counts())

Country_Group
Germany           1914
Italy             1793
France            1229
Switzerland       1112
Netherlands       1056
Croatia            811
Spain              702
Other              265
United Kingdom     215
Denmark            172
Portugal           135
Austria             88
Greece              83
Malta               72
Turkey              64
Belgium             62
Poland              41
Slovenia            39
Finland             35
Name: count, dtype: int64


In [ ]:
conditions = {
    "Used boat": "Used",
    "new boat from stock": "New",
    "new boat on order": "New",
    "Display Model": "Display model",
}

parts = df["Type"].str.split(",")

df["Condition"] = parts.str[0].map(conditions).fillna("Unknown")

last = parts.str[-1]
df["Fuel"] = last.where(~last.isin(list(conditions) + [""]), "Unknown")

print(df["Condition"].value_counts())
print(df["Fuel"].value_counts())

Condition
Used             7327
New              2356
Display model     119
Unknown            86
Name: count, dtype: int64
Fuel
Diesel      4568
Unleaded    3040
Unknown     2212
Electric      52
Gas           13
Hybrid         2
Propane        1
Name: count, dtype: int64


In [10]:
df["Fuel"] = df["Fuel"].replace({"Propane": "Gas"})

In [ ]:
df["Boat_Type_Main"] = df["Boat Type"].str.split(",").str[0].str.strip()

df["Age"] = 2021 - df["Year Built"]

df["Price_per_m"] = (df["Price_EUR"] / df["Length"]).round(0)

print(df["Boat_Type_Main"].value_counts().head(10))
print(df[["Year Built", "Age", "Price_EUR", "Length", "Price_per_m"]].head(8))

Boat_Type_Main
Motor Yacht            2736
Sport Boat             1417
Flybridge              1203
Trawler                 679
Cabin Boat              654
Pilothouse              614
Hardtop                 513
Center console boat     377
Bowrider                310
Deck Boat               285
Name: count, dtype: int64
   Year Built   Age  Price_EUR  Length  Price_per_m
0        2017     4     3087.0    4.00        772.0
1        2020     1     3490.0    4.00        872.0
2        <NA>  <NA>     3487.0    3.69        945.0
3        2020     1     3484.0    3.00       1161.0
4        2019     2     3399.0    3.55        957.0
5        <NA>  <NA>     3376.0    4.03        838.0
6        1999    22     3330.0    6.20        537.0
7        <NA>  <NA>     3336.0    3.00       1112.0


In [ ]:
df["Material"] = df["Material"].replace({"": "Unknown"})

df = df.rename(columns={"Number of views last 7 days": "Views_7d"})

cols = [
    "Boat_Type_Main", "Manufacturer", "Condition", "Fuel", "Material",
    "Year Built", "Age", "Length", "Width",
    "Price_EUR", "Price_per_m", "Currency",
    "Country", "Country_Group", "Location",
    "Views_7d",
]
df_clean = df[cols]

print(df_clean.shape)
print(df_clean.head())

df_clean.to_csv("boats_clean.csv", index=False, encoding="utf-8-sig")
files.download("boats_clean.csv")

(9888, 16)
        Boat_Type_Main          Manufacturer Condition     Fuel  \
0          Motor Yacht  Rigiflex power boats       New  Unknown   
1  Center console boat     Terhi power boats       New  Unknown   
2           Sport Boat    Marine power boats       New  Unknown   
3           Sport Boat    Pioner power boats       New  Unknown   
4         Fishing Boat    Linder power boats       New  Unknown   

        Material  Year Built   Age  Length  Width  Price_EUR  Price_per_m  \
0        Unknown        2017     4    4.00   1.90     3087.0        772.0   
1  Thermoplastic        2020     1    4.00   1.50     3490.0        872.0   
2      Aluminium        <NA>  <NA>    3.69   1.42     3487.0        945.0   
3        Unknown        2020     1    3.00   1.00     3484.0       1161.0   
4      Aluminium        2019     2    3.55   1.46     3399.0        957.0   

  Currency      Country Country_Group  \
0      CHF  Switzerland   Switzerland   
1      EUR      Germany       Germany   


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>